In [ ]:
from sqlalchemy import create_engine
import pandas as pd

def create_db_engine(user, password, host="localhost", port=3306, database="sakila"):
    connection_string = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
    engine = create_engine(connection_string)
    return engine


In [ ]:
engine = create_db_engine("root", "your_password")


In [ ]:
def rentals_month(engine, month, year):
    query = f"""
        SELECT
            rental_id,
            customer_id,
            rental_date
        FROM rental
        WHERE MONTH(rental_date) = {month}
          AND YEAR(rental_date) = {year};
    """
    
    df = pd.read_sql(query, engine)
    return df


In [ ]:
def rental_count_month(df, month, year):
    
    column_name = f"rentals_{month:02d}_{year}"
    
    rental_count = (
        df.groupby("customer_id")
          .size()
          .reset_index(name=column_name)
    )
    
    return rental_count


In [ ]:
def compare_rentals(df1, df2):
    
    combined = pd.merge(
        df1,
        df2,
        on="customer_id",
        how="outer"
    ).fillna(0)
    
    # Identificar automaticamente colunas de rentals
    rental_cols = [col for col in combined.columns if col.startswith("rentals_")]
    
    combined["difference"] = combined[rental_cols[1]] - combined[rental_cols[0]]
    
    return combined
